# Counting raw reads

Try to determine the sequencing depth for each bacteria sample: using `seqkit stats` 
to count the number of reads in each FASTQ file. The `-j4` option allows for parallel 
processing using 4 threads.

```bash
seqkit stats -j4 data/240926_M04028_0172_000000000-LN722/*.gz > data/seq_stats.csv
```

In [ ]:
from pathlib import Path

import pandas as pd

Now open the resulting `seq_stats.csv` file in Python and merge it with the metadata 
to get a complete overview of the samples and their sequencing depth.

In [ ]:
data_dir = Path("../data")
seq_stats = pd.read_fwf(data_dir / "seq_stats.csv", colspecs="infer", thousands=",")
seq_stats["num_seqs"] = seq_stats["num_seqs"].str.replace(",", "").astype(int)
seq_stats.head()

In [ ]:
seq_stats.info()

In [ ]:
samplesheet_bacteria = pd.read_csv(data_dir / "samplesheet_bacteria.csv")
samplesheet_bacteria.head()

In [ ]:
# Prepare samplesheet for melting
id_vars = [c for c in samplesheet_bacteria.columns if c not in ("forwardReads", "reverseReads")]
samplesheet_bacteria_files = samplesheet_bacteria.melt(
    id_vars=id_vars,
    value_vars=["forwardReads", "reverseReads"],
    var_name="read",
    value_name="file"
).assign(read=lambda df: df["read"].str.replace("Reads", "", regex=False).str.replace("forward", "forward", regex=False).str.replace("reverse", "reverse", regex=False))

samplesheet_bacteria_files.head()

In [ ]:
metadata_bacteria = pd.read_table(data_dir / "metadata_bacteria.tsv")
metadata_bacteria.head()

Join the sequencing stats with the metadata:

In [ ]:
samplesheet_bacteria_merged = samplesheet_bacteria_files.merge(
    metadata_bacteria,
    on="sampleID",
    how="left",
    validate="m:1"
)

samplesheet_bacteria_merged.head()

In [ ]:
samplesheet_bacteria_full = samplesheet_bacteria_merged.merge(
    seq_stats,
    on="file",
    how="inner"
)
samplesheet_bacteria_full.head()

ok now take only "nov 23" samples:

In [ ]:
nov_23 = samplesheet_bacteria_full[samplesheet_bacteria_full["date"] == "nov 23"]
nov_23.head()

In [ ]:
print(f"Got {len(nov_23)} paired files for nov 23")
print(f"Total number of sequences: {nov_23['num_seqs'].sum()}")

In [ ]:
# convert to float then calculate the averages
s_sum = nov_23["sum_len"].astype(float).sum()
s_n = nov_23["num_seqs"].astype(float).sum()
overall_avg = s_sum / s_n

mean_of_avg = nov_23["avg_len"].astype(float).mean()
weighted_avg = (nov_23["avg_len"].astype(float) * nov_23["num_seqs"].astype(float)).sum() / s_n

means = {
    "overall_avg": overall_avg,
    "mean_of_avg": mean_of_avg,
    "weighted_avg": weighted_avg,
    "total_sum_len": s_sum,
    "total_num_seqs": s_n
}

print(f"Weighted average length: {int(means['weighted_avg'])} bp")